# Human Action Recognition: Model Comparison, Tuning and Explainable AI

This notebook applies the teacher's feedback to the project:

| Feedback | What this notebook does |
|---|---|
| Dropout 0.05 | `DROPOUT = 0.05`, applied before the classifier in every model |
| Fine-tune for 20–25 epochs | `EPOCHS = 25` (the maximum) |
| Early stopping, patience 3 | `PATIENCE = 3`, watching a **validation set** (10% of the training photos), never the test set |
| Far fewer hyperparameters | Only the settings in **one cell** (Part 0); everything else is fixed in the training code |
| Weight decay | `WEIGHT_DECAY = 4e-4`, plus an optional run at `1e-3` to show its effect |
| Use various models and compare | **Four models trained exactly the same way**, with a results table and comparison charts |
| Check the model and adjust it | Optional runs: **no dropout**, **higher weight decay**, and a **smaller model** (about 4× fewer weights) |
| The curves are not good | Train, validation and test are measured **the same way every epoch** (clean photos, no dropout), so the gap shows overfitting, and the **best epoch is marked** |
| Explainable AI | Confusion matrix, **occlusion maps** (which part of the photo matters), **attention maps** for the transformer, and examples of the mistakes |

> **On Kaggle:** set **Accelerator → GPU T4 x2** and **Internet → On**, attach the *Human Action Recognition (HAR) Dataset*,
> then use **Save Version → Save & Run All**. All runs together take about **1.5–2 hours**; set `RUN_OPTIONAL = False` to save about 40 minutes.

# Part 0 — Settings
These are the **only** settings. The learning rate is small (2e-5) because the pretrained weights are already good:
a larger step (5e-5) made the CLIP transformer lose its pretrained knowledge in the first epochs (its validation accuracy
fell from 79.9% to 72.7% after epoch 1). Everything else (batch size 32, image size 224, AdamW, 1 warm-up epoch then cosine decay,
label smoothing 0.1, gradient clipping) is fixed inside the training code in Part 2.

In [ ]:
DROPOUT = 0.05          # dropout before the classifier, in every model
EPOCHS = 25             # maximum number of epochs; early stopping usually ends sooner
PATIENCE = 3            # stop after 3 epochs without a better validation accuracy
LEARNING_RATE = 2e-5    # AdamW learning rate for the pretrained layers (the new classifier layer uses 10x)
WEIGHT_DECAY = 4e-4     # AdamW weight decay

RUN_OPTIONAL = True     # extra check runs: weight decay 1e-3, no dropout, smaller model

# Part 1 — Data

In [ ]:
import os, gc, glob, copy, math, time, random, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", message="Detected call of")
SEED = 42
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Device: ", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU - turn on the GPU accelerator!")

csv_matches = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)
assert csv_matches, "Dataset not found - click 'Add Input' and attach the Human Action Recognition dataset."
DATA_DIR = os.path.dirname(csv_matches[0])
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
OUTPUT_DIR = "/kaggle/working"

df = pd.read_csv(os.path.join(DATA_DIR, "Training_set.csv"))
class_names = sorted(df["label"].unique())
df["label_id"] = df["label"].map({n: i for i, n in enumerate(class_names)})
NUM_CLASSES = len(class_names)
print(f"{len(df)} labeled photos, {NUM_CLASSES} actions")

## 1.1 Group near-duplicates and split the data
Near-duplicate photos (same dHash fingerprint, at most 4 of 64 bits different) are kept on the same side of every split,
so the test score can't be inflated by copies (see notebook 03).

- **Test 10%** – used only to report results.
- **Validation 10% of the rest** – watched by early stopping and used to pick the best epoch.
  (5% was only ~560 photos: too noisy, so early stopping could stop by chance.)
- **Training** – everything else.

In [ ]:
def dhash(path):
    px = np.asarray(Image.open(path).convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return (px[:, 1:] > px[:, :-1]).flatten()

with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    bits = np.array(list(pool.map(dhash, [os.path.join(TRAIN_IMG_DIR, f) for f in df["filename"]])), dtype=np.float32)
pa, pb = [], []
for s in range(0, len(bits), 1024):
    c = bits[s:s + 1024]
    a, b = np.nonzero(64 - (c @ bits.T + (1 - c) @ (1 - bits).T) <= 4)
    a = a + s; keep = b > a; pa.append(a[keep]); pb.append(b[keep])
pa, pb = np.concatenate(pa), np.concatenate(pb)
_, groups = connected_components(coo_matrix((np.ones(len(pa)), (pa, pb)), shape=(len(df), len(df))), directed=False)
df["group"] = groups

y_all, all_idx = df["label_id"].values, np.arange(len(df))
trval_idx, test_idx = next(StratifiedGroupKFold(10, shuffle=True, random_state=SEED).split(all_idx, y_all, groups))
tr, va = next(StratifiedGroupKFold(10, shuffle=True, random_state=SEED).split(trval_idx, y_all[trval_idx], groups[trval_idx]))
train_df = df.iloc[trval_idx[tr]].reset_index(drop=True)
val_df = df.iloc[trval_idx[va]].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)
assert not (set(train_df.group) & set(val_df.group) or set(train_df.group) & set(test_df.group) or set(val_df.group) & set(test_df.group))
# a fixed sample of training photos, measured exactly like validation and test (for the curves)
train_eval_df = train_df.sample(min(2000, len(train_df)), random_state=SEED).reset_index(drop=True)
print(f"Training {len(train_df)} | validation {len(val_df)} | test {len(test_df)} photos")

# Part 2 — Fixed training code
Nothing in this cell needs changing. For every model it does the same thing:

1. Load the pretrained model, replace its classifier with **dropout (`DROPOUT`) → dense layer with 15 outputs**.
2. Train with **AdamW** (`LEARNING_RATE`, `WEIGHT_DECAY`), 1 warm-up epoch then cosine decay over `EPOCHS`,
   label smoothing 0.1, mixed precision and gradient clipping.
3. After **every** epoch, measure loss and accuracy on a training sample, the validation set and the test set
   **the same way** (clean photos, dropout off). Only the validation accuracy is used for decisions.
4. **Early stopping:** stop when validation accuracy hasn't improved for `PATIENCE` epochs, and keep the best epoch's weights.

In [ ]:
BATCH_SIZE, IMG_SIZE, LABEL_SMOOTHING, GRAD_CLIP, HEAD_LR_MULT = 32, 224, 0.1, 1.0, 10
NUM_WORKERS = os.cpu_count() or 2

class Photos(Dataset):
    def __init__(self, frame, tf):
        self.files, self.labels, self.tf = frame["filename"].tolist(), frame["label_id"].tolist(), tf
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        return self.tf(Image.open(os.path.join(TRAIN_IMG_DIR, self.files[i])).convert("RGB")), self.labels[i]

def make_transforms(mean, std):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
        transforms.ColorJitter(0.2, 0.2), transforms.ToTensor(), transforms.Normalize(mean, std)])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    return train_tf, eval_tf

def loader(frame, tf, train):
    return DataLoader(Photos(frame, tf), batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=use_amp)

@torch.no_grad()
def predict(net, dl):
    net.eval()
    probs, labels = [], []
    for x, y in dl:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            probs.append(net(x.to(device, non_blocking=True)).float().softmax(1).cpu())
        labels.append(y)
    return torch.cat(probs), torch.cat(labels)

def measure(net, dl):
    p, y = predict(net, dl)
    return F.nll_loss(torch.log(p + 1e-9), y).item(), (p.argmax(1) == y).float().mean().item()

def build_model(name, dropout, pretrained=True):
    net = timm.create_model(name, pretrained=pretrained, num_classes=NUM_CLASSES, drop_rate=dropout)
    cfg = timm.data.resolve_model_data_config(net)
    return net.to(device), cfg["mean"], cfg["std"]

def train_model(name, label, dropout=None, weight_decay=None):
    dropout = DROPOUT if dropout is None else dropout
    weight_decay = WEIGHT_DECAY if weight_decay is None else weight_decay
    seed_everything(SEED)
    net, mean, std = build_model(name, dropout)
    train_tf, eval_tf = make_transforms(mean, std)
    train_dl = loader(train_df, train_tf, True)
    eval_dls = {s: loader(f, eval_tf, False) for s, f in (("train", train_eval_df), ("val", val_df), ("test", test_df))}
    head = set(id(p) for p in net.get_classifier().parameters())
    opt = torch.optim.AdamW([
        {"params": [p for p in net.parameters() if id(p) not in head], "lr": LEARNING_RATE},
        {"params": [p for p in net.parameters() if id(p) in head], "lr": LEARNING_RATE * HEAD_LR_MULT}],
        weight_decay=weight_decay)
    total, warm = EPOCHS * len(train_dl), len(train_dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(
        s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))), 0.01))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    history, best, best_state, bad, t_start = [], -1.0, None, 0, time.time()
    for epoch in range(1, EPOCHS + 1):
        net.train()
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(net(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
        row = {"epoch": epoch}
        for s, dl in eval_dls.items():
            row[f"{s}_loss"], row[f"{s}_acc"] = measure(net, dl)
        history.append(row)
        improved = row["val_acc"] > best
        if improved:
            best, bad = row["val_acc"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
        print(f"[{label}] epoch {epoch:2d} | train {row['train_acc']:.1%} | val {row['val_acc']:.1%} | "
              f"test {row['test_acc']:.1%}{'  <- best' if improved else ''}")
        if bad >= PATIENCE:
            print(f"[{label}] early stopping: no better validation accuracy for {PATIENCE} epochs")
            break
    net.load_state_dict(best_state)
    h = pd.DataFrame(history)
    b = h.loc[h["val_acc"].idxmax()]
    result = {"model": label, "timm name": name, "parameters (M)": round(sum(p.numel() for p in net.parameters()) / 1e6, 1),
              "dropout": dropout, "weight decay": weight_decay, "epochs run": len(h), "best epoch": int(b["epoch"]),
              "train acc": b["train_acc"], "val acc": b["val_acc"], "test acc": b["test_acc"],
              "train-val gap": b["train_acc"] - b["val_acc"], "minutes": round((time.time() - t_start) / 60, 1)}
    del opt, scaler, train_dl
    return net, eval_tf, h, result

def free(net):
    net.cpu(); del net; gc.collect()
    if use_amp:
        torch.cuda.empty_cache()

# Part 3 — Four models, trained the same way
| Model | Type | Pretrained on |
|---|---|---|
| EfficientNet-B0 | small CNN (like our first model) | ImageNet-1k photos |
| ResNet-50 | classic CNN with skip connections | ImageNet-1k photos |
| ConvNeXt-Tiny | modern CNN | ImageNet-22k photos |
| CLIP ViT-B/16 | Vision Transformer | 400M image–caption pairs |

Same data, same settings, same early stopping. Only the network changes.

In [ ]:
MODELS = {
    "EfficientNet-B0": "efficientnet_b0.ra_in1k",
    "ResNet-50": "resnet50.a1_in1k",
    "ConvNeXt-Tiny": "convnext_tiny.fb_in22k_ft_in1k",
    "CLIP ViT-B/16": "vit_base_patch16_clip_224.openai",
}
results, histories, kept = [], {}, {}
for label, name in MODELS.items():
    net, eval_tf, h, r = train_model(name, label)
    results.append(r); histories[label] = h
    kept[label] = ({k: v.cpu() for k, v in net.state_dict().items()}, eval_tf)   # keep weights for Part 6
    free(net)

# Part 4 — Results
## 4.1 Comparison table and chart
Accuracies are taken at each model's **best epoch** (highest validation accuracy).
The **train–validation gap** shows overfitting: the bigger it is, the more the model memorised the training photos.

In [ ]:
res = pd.DataFrame(results)
show = res[["model", "parameters (M)", "epochs run", "best epoch", "train acc", "val acc", "test acc", "train-val gap", "minutes"]]
display(show.style.format({"train acc": "{:.1%}", "val acc": "{:.1%}", "test acc": "{:.1%}", "train-val gap": "{:+.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.arange(len(res))
for k, (col, lab) in enumerate((("train acc", "train"), ("val acc", "validation"), ("test acc", "test"))):
    bars = axes[0].bar(x + (k - 1) * 0.27, res[col] * 100, 0.27, label=lab)
    if col == "test acc":
        axes[0].bar_label(bars, labels=[f"{v:.1%}" for v in res[col]], padding=2, fontsize=9)
axes[0].set_xticks(x); axes[0].set_xticklabels(res["model"]); axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Accuracy at best epoch (%)"); axes[0].set_title("Accuracy per model"); axes[0].legend(loc="lower right")
axes[1].scatter(res["parameters (M)"], res["test acc"] * 100, s=80)
for _, r in res.iterrows():
    axes[1].annotate(r["model"], (r["parameters (M)"], r["test acc"] * 100), xytext=(6, 4), textcoords="offset points")
axes[1].set_xlabel("Parameters (millions)"); axes[1].set_ylabel("Test accuracy (%)"); axes[1].set_title("Size vs. accuracy")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
res.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)

## 4.2 Training curves
All three lines are measured the same way after every epoch: clean photos, dropout off.
- **Train above validation, gap growing** → overfitting.
- **Validation flat for 3 epochs** → early stopping ends training.
- The **dashed line** marks the best epoch, whose weights are kept. Test is shown for information only; it is never used for decisions.

In [ ]:
def plot_curves(hists, title):
    n = len(hists)
    fig, axes = plt.subplots(2, n, figsize=(4.6 * n, 7.5), squeeze=False)
    for j, (label, h) in enumerate(hists.items()):
        best = int(h.loc[h["val_acc"].idxmax(), "epoch"])
        for i, (metric, name) in enumerate((("acc", "Accuracy"), ("loss", "Loss"))):
            ax = axes[i, j]
            for s, style in (("train", "o-"), ("val", "s-"), ("test", "^--")):
                vals = h[f"{s}_{metric}"] * (100 if metric == "acc" else 1)
                ax.plot(h["epoch"], vals, style, ms=4, label={"train": "train", "val": "validation", "test": "test"}[s])
            ax.axvline(best, color="gray", linestyle="--", linewidth=1)
            ax.set_title(f"{label}: {name.lower()}" + (f" (best epoch {best})" if i == 0 else ""), fontsize=10)
            ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
            if j == 0:
                ax.set_ylabel(name + (" (%)" if metric == "acc" else ""))
            if i == 0 and j == 0:
                ax.legend(fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_curves(histories, "Same measurement for train, validation and test after every epoch")

# Part 5 — Checking and adjusting the model (optional runs)
Each run changes **one thing** compared with the CLIP ViT-B/16 run in Part 3, so its effect is clear:

| Run | What changes | Question it answers |
|---|---|---|
| Weight decay 1e-3 | `WEIGHT_DECAY` 4e-4 → 1e-3 | Does stronger weight decay reduce overfitting or hurt accuracy? |
| No dropout | `DROPOUT` 0.05 → 0 | Does the small dropout help? |
| Smaller model | ViT-B/16 (86M) → ViT-S/16 (22M, about 4× fewer weights) | Do we need the big model? |

In [ ]:
opt_results, opt_hist = [], {}
if RUN_OPTIONAL:
    runs = [("ViT-B/16, weight decay 1e-3", MODELS["CLIP ViT-B/16"], DROPOUT, 1e-3),
            ("ViT-B/16, no dropout", MODELS["CLIP ViT-B/16"], 0.0, WEIGHT_DECAY),
            ("ViT-S/16 (smaller)", "vit_small_patch16_224.augreg_in21k_ft_in1k", DROPOUT, WEIGHT_DECAY)]
    for label, name, dr, wd in runs:
        net, _, h, r = train_model(name, label, dropout=dr, weight_decay=wd)
        opt_results.append(r); opt_hist[label] = h
        free(net)
    base = res[res["model"] == "CLIP ViT-B/16"]
    adj = pd.concat([base, pd.DataFrame(opt_results)], ignore_index=True)
    display(adj[["model", "parameters (M)", "dropout", "weight decay", "best epoch", "train acc", "val acc", "test acc", "train-val gap"]]
            .style.format({"train acc": "{:.1%}", "val acc": "{:.1%}", "test acc": "{:.1%}", "train-val gap": "{:+.1%}"}))
    plot_curves(opt_hist, "Optional runs")
    adj.to_csv(os.path.join(OUTPUT_DIR, "adjustment_runs.csv"), index=False)
else:
    print("Optional runs skipped (RUN_OPTIONAL = False).")

# Part 6 — Explainable AI
We explain the **best model** (highest validation accuracy in Part 3):

1. **Confusion matrix and per-action scores** – *which* actions it gets wrong.
2. **Occlusion maps** – *where* it looks: a grey square slides over the photo, and the drop in the probability of the predicted action
   is measured. Red areas are the ones the prediction depends on. This works for any model.
3. **Attention maps** (transformer only) – *what the [CLS] token attends to*, combined over all 12 layers (attention rollout).
4. **Mistakes** – wrong predictions with their occlusion maps, to see *why* the model got confused.

In [ ]:
best_label = res.loc[res["val acc"].idxmax(), "model"]
state, eval_tf = kept[best_label]
xai_model, _, _ = build_model(MODELS[best_label], DROPOUT, pretrained=False)
xai_model.load_state_dict(state); xai_model.eval()
print("Explaining:", best_label)

probs, labels = predict(xai_model, loader(test_df, eval_tf, False))
pred = probs.argmax(1).numpy(); true = labels.numpy()
print(classification_report(true, pred, target_names=class_names, digits=3, zero_division=0))
cm = confusion_matrix(true, pred, labels=range(NUM_CLASSES))
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=60, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"{best_label}: confusion matrix (colour = share of each true action)")
fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

In [ ]:
MEAN = torch.tensor(timm.data.resolve_model_data_config(xai_model)["mean"]).view(3, 1, 1)
STD = torch.tensor(timm.data.resolve_model_data_config(xai_model)["std"]).view(3, 1, 1)
def to_img(t):
    return (t * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

@torch.no_grad()
def occlusion_map(x, target, size=32, stride=16):
    # probability drop of `target` when a grey square covers each position
    xs, pos = [], []
    for top in range(0, IMG_SIZE - size + 1, stride):
        for left in range(0, IMG_SIZE - size + 1, stride):
            xo = x.clone(); xo[:, top:top + size, left:left + size] = 0   # 0 = the average colour after normalization
            xs.append(xo); pos.append((top, left))
    base = xai_model(x[None].to(device)).float().softmax(1)[0, target].item()
    drops = []
    for i in range(0, len(xs), 64):
        p = xai_model(torch.stack(xs[i:i + 64]).to(device)).float().softmax(1)[:, target].cpu()
        drops.append(base - p)
    drops = torch.cat(drops)
    heat, count = torch.zeros(IMG_SIZE, IMG_SIZE), torch.zeros(IMG_SIZE, IMG_SIZE)
    for d, (top, left) in zip(drops, pos):
        heat[top:top + size, left:left + size] += d; count[top:top + size, left:left + size] += 1
    return (heat / count.clamp(min=1)).clamp(min=0).numpy()

def show_maps(rows, title):
    fig, axes = plt.subplots(len(rows), 3, figsize=(11, 3.6 * len(rows)), squeeze=False)
    for r, i in enumerate(rows):
        x, y = eval_tf(Image.open(os.path.join(TRAIN_IMG_DIR, test_df["filename"][i])).convert("RGB")), int(true[i])
        p = int(pred[i]); heat = occlusion_map(x, p)
        axes[r, 0].imshow(to_img(x)); axes[r, 0].set_title(f"true: {class_names[y]}", fontsize=10)
        axes[r, 1].imshow(to_img(x)); axes[r, 1].imshow(heat, cmap="jet", alpha=0.5)
        axes[r, 1].set_title(f"predicted: {class_names[p]} ({probs[i, p]:.0%})", fontsize=10)
        axes[r, 2].imshow(heat, cmap="jet"); axes[r, 2].set_title("occlusion map (red = important)", fontsize=10)
        for a in axes[r]: a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED)
correct = np.where(pred == true)[0]
wrong = np.where(pred != true)[0]
show_maps(rng.choice(correct, size=min(6, len(correct)), replace=False), "Correct predictions: where the model looks")

In [ ]:
def attention_rollout(x):
    # Combines the [CLS] attention of all transformer layers (Abnar & Zuidema, 2020)
    store = []
    hooks = []
    for blk in xai_model.blocks:
        blk.attn.fused_attn = False
        hooks.append(blk.attn.attn_drop.register_forward_hook(lambda m, i, o: store.append(i[0].float().mean(1)[0].cpu())))
    with torch.no_grad():
        xai_model(x[None].to(device))
    for h in hooks:
        h.remove()
    roll = torch.eye(store[0].shape[-1])
    for a in store:
        a = a + torch.eye(a.shape[-1]); a = a / a.sum(-1, keepdim=True)
        roll = a @ roll
    n = int(round((roll.shape[-1] - 1) ** 0.5))
    m = roll[0, 1:1 + n * n].reshape(1, 1, n, n)
    return F.interpolate(m, size=(IMG_SIZE, IMG_SIZE), mode="bilinear")[0, 0].numpy()

if hasattr(xai_model, "blocks"):
    sample = rng.choice(correct, size=min(6, len(correct)), replace=False)
    fig, axes = plt.subplots(2, len(sample), figsize=(3.2 * len(sample), 6.6), squeeze=False)
    for c, i in enumerate(sample):
        x = eval_tf(Image.open(os.path.join(TRAIN_IMG_DIR, test_df["filename"][i])).convert("RGB"))
        axes[0, c].imshow(to_img(x)); axes[0, c].set_title(class_names[int(pred[i])], fontsize=10)
        axes[1, c].imshow(to_img(x)); axes[1, c].imshow(attention_rollout(x), cmap="jet", alpha=0.5)
        axes[0, c].axis("off"); axes[1, c].axis("off")
    fig.suptitle("Attention rollout: what the transformer attends to (bottom row)"); plt.tight_layout(); plt.show()
else:
    print("The best model is a CNN, so there are no attention maps; the occlusion maps above explain it.")

In [ ]:
pairs = sorted(((cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j), reverse=True)[:5]
print("Most common mix-ups:")
for n, i, j in pairs:
    print(f"  {class_names[i]:>20} predicted as {class_names[j]:<20} {n} times")
examples = [k for _, i, j in pairs[:3] for k in wrong if true[k] == i and pred[k] == j][:6]
if examples:
    show_maps(examples, "Mistakes: what the model looked at when it got it wrong")

# How sure is the model? Confidence of correct vs. wrong predictions
conf = probs.max(1).values.numpy()
plt.figure(figsize=(7, 3.5))
plt.hist(conf[pred == true], bins=20, range=(0, 1), alpha=0.7, label="correct")
plt.hist(conf[pred != true], bins=20, range=(0, 1), alpha=0.7, label="wrong")
plt.xlabel("Confidence of the prediction"); plt.ylabel("Test photos"); plt.legend()
plt.title("Wrong predictions are usually less confident"); plt.tight_layout(); plt.show()

# Summary
- **Settings:** dropout 0.05, up to 25 epochs, early stopping with patience 3 on a validation set, weight decay 4e-4,
  one learning rate. Everything else is fixed.
- **Comparison:** four models trained identically (Part 4 table and charts).
- **Adjustments:** the effect of weight decay, dropout and model size (Part 5).
- **Curves:** train, validation and test measured the same way, best epoch marked (Part 4.2).
- **Explainable AI:** confusion matrix, occlusion maps, attention maps and mistakes (Part 6).